# Universal motifs (U1–U5)
Do motifs learned in one mouse / brain region / LFP translate to other mice, regions and the EEG, and are they
grounded in spiking? Awake references, stimulation excluded, locally re-referenced channels (CSD on probes,
common average on EEG). Spatial mouse split (fixed 2026-11-16): every decision (number of motifs, motif shapes,
sign of each relationship) is made on the discovery mice (688277, 692644, 703063, 709401); the confirmation mice
(703064, 703065, 720762) test it, one-sided in the discovery direction.

Inputs: per-channel awake motif libraries (every ~160 µm on probes, every EEG electrode; 768 channels with a
library) and per-channel residual covariances (2,426 grey-matter channels and EEG electrodes).

Code: `src/universal.py`, runner `src/run_universal.py`; results in `data/processed/universal/`.

In [1]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")
REPO = Path("/Users/namanrudrakshi/Developer/dc/dc-allen-data-acquire")
sys.path.insert(0, str(REPO / "code" / "python"))
from src import awake_grounding as ag, universal as uv
from src.state_motifs import BANDS
PU = REPO / "data/processed/universal"
OUT = REPO / "data" / "out" / "2026-11-16_universal_motifs"; FIG = OUT / "figures"; FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "0.4", "axes.labelcolor": "0.15", "xtick.color": "0.3", "ytick.color": "0.3"})
def save(fig, name): fig.savefig(FIG / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
SP = json.load(open(REPO / "data/processed/mouse_split.json"))["spatial"]
DISC_C, CONF_C = "#2a78d6", "#1baf7a"
BANDLAB = [f"{int(a)}–{int(b)}" for a, b in BANDS]
U1 = ag.load(PU / "u1.pkl"); D = U1["D"]
NAMES = ["M1 broadband", "M2 low vs gamma", "M3 slow vs beta"]

## U1. A candidate universal set
Sign-free k-lines clustering of the pooled awake templates (each mouse weighted equally). Rule fixed before running:
keep the k-th motif if, in ≥ 3 of 4 held-out discovery mice, it adds more held-out variance than a random smooth
direction (smoothness matched to the templates) and it reappears (|cos| ≥ 0.9) in ≥ 3 of 4 leave-one-mouse-out fits.

In [2]:
pm = U1["per_mouse"]
print("k chosen:", U1["chosen"])
pm.assign(beats_random=pm.gain > pm.random_gain).round(3)

k chosen: 3


,k,mouse,gain,random_gain,captured,own,fold_cos_kth,beats_random
0,1,688277,0.185,0.153,0.185,0.220,1.000,True
1,1,692644,0.293,0.181,0.293,0.332,0.997,True
2,1,703063,0.293,0.181,0.293,0.318,0.981,True
3,1,709401,0.267,0.187,0.267,0.312,0.976,True
4,2,688277,0.143,0.125,0.334,0.385,1.000,True
5,2,692644,0.142,0.121,0.448,0.497,0.987,True
6,2,703063,0.136,0.121,0.436,0.479,0.942,True
7,2,709401,0.165,0.125,0.432,0.462,0.998,True
8,3,688277,0.123,0.108,0.462,0.520,1.000,True
9,3,692644,0.115,0.104,0.570,0.615,0.984,True


Shapes (discovery fit) against an independent fit on the confirmation mice, and how close each motif is to plain
spectral descriptors (cos with a flat all-band vector and with spectral tilt, the 1/f-slope proxy).

In [3]:
c = U1["confirm"]
rng = np.random.default_rng(1)
null = np.array([uv.match(D, uv.smooth_directions(3, U1["ell"], rng))[0] for _ in range(2000)])
shape = pd.DataFrame({"motif": NAMES, "share of templates (discovery)": U1["folds"][3]["share"],
                      "|cos| with confirmation fit": c["match_cos"],
                      "p (pre-set: vs best of 3 random)": c["match_p"],
                      "p (per motif, post hoc)": [(1 + (null[:, i] >= c["match_cos"][i]).sum()) / 2001 for i in range(3)],
                      "cos with flat": D @ uv.unit(np.ones(10)), "cos with tilt": D @ uv.TILT})
shape.to_csv(OUT / "U1_shapes.csv", index=False)
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for i, ax in enumerate(axs):
    ax.plot(range(10), D[i], "-o", color=DISC_C, label="discovery mice")
    ax.plot(range(10), c["D_conf"][i], "-o", color=CONF_C, label="confirmation mice (own fit)")
    ax.axhline(0, color="0.6", lw=0.6); ax.set_title(f"{NAMES[i]}  (|cos| {c['match_cos'][i]:.2f})", fontsize=9, loc="left")
    ax.set_xticks(range(10), BANDLAB, rotation=60, fontsize=7); ax.set_xlabel("band (Hz)")
axs[0].set_ylabel("weight"); axs[0].legend(frameon=False, fontsize=7)
save(fig, "U1_motif_shapes"); plt.show()
shape.round(3)

,motif,share of templates (discovery),|cos| with confirmation fit,p (pre-set: vs best of 3 random),"p (per motif, post hoc)",cos with flat,cos with tilt
0,M1 broadband,0.508,0.999,0.001,0.000,0.985,-0.009
1,M2 low vs gamma,0.323,0.887,0.449,0.141,-0.267,0.713
2,M3 slow vs beta,0.169,0.415,0.999,0.824,-0.289,-0.616


Held-out reconstruction: share of each held-out channel's top-3 residual variance captured by the 3 motifs
(1 = as good as the channel's own top 3 principal directions), against the training mice's pooled band PCA and
random smooth directions. Discovery mice use their leave-one-mouse-out set; confirmation mice the discovery set.

In [4]:
H = pd.concat([U1["heldout_disc"].assign(group="discovery (held out)"), c["reconstruction"].assign(group="confirmation")])
H["kind"] = np.where(H.is_eeg, "EEG", "LFP")
rec = H.groupby(["group", "kind"])[["dictionary", "pooled PCA", "random smooth"]].median()
rec.to_csv(OUT / "U1_reconstruction.csv"); rec.round(3)

dictionary  pooled PCA  random smooth
group                kind                                       
confirmation         EEG        0.921       0.893          0.773
                     LFP        0.954       0.958          0.845
discovery (held out) EEG        0.938       0.934          0.793
                     LFP        0.941       0.948          0.827

In [5]:
W = U1["where"]
live = W.pivot_table(index="class", columns="motif", values="w", aggfunc="sum").fillna(0)
live = live.div(live.sum(1), axis=0); live.columns = NAMES
live.round(2)

,M1 broadband,M2 low vs gamma,M3 slow vs beta
class,,,
EEG,0.36,0.48,0.16
cortex,0.68,0.18,0.14
hippocampus,0.64,0.32,0.04
midbrain,0.75,0.00,0.25
olfactory,1.00,0.00,0.00
other,1.00,0.00,0.00
striatum / septum / pallidum,0.35,0.29,0.37
thalamus,0.54,0.32,0.14


## U2. Transfer between mice, regions and modalities
Score = share of a target channel's top-direction variance captured by the source channel's best template
(1 = perfect; target's own library ≈ 0.95; random smooth direction ≈ 0.5–0.6). Medians per kind of pair.

In [6]:
U2 = ag.load(PU / "u2.pkl"); P = U2["pairs"]
P["grp"] = np.where(P.src_mouse.isin(SP["discovery"]) & P.tgt_mouse.isin(SP["discovery"]), "discovery",
           np.where(P.src_mouse.isin(SP["confirmation"]) & P.tgt_mouse.isin(SP["confirmation"]), "confirmation", "across groups"))
lfp = (P.src_kind == "LFP") & (P.tgt_kind == "LFP")
P["pair"] = np.select(
    [lfp & P.same_array & (P.depth_gap_um <= 400), lfp & P.same_array, lfp & P.same_mouse, lfp & P.same_area, lfp & P.same_class, lfp,
     (P.src_kind == "LFP") & (P.tgt_kind == "EEG") & P.same_mouse, (P.src_kind == "LFP") & (P.tgt_kind == "EEG"),
     (P.src_kind == "EEG") & (P.tgt_kind == "EEG") & P.same_mouse, (P.src_kind == "EEG") & (P.tgt_kind == "EEG"),
     (P.src_kind == "EEG") & P.same_mouse, P.src_kind == "EEG"],
    ["LFP→LFP same probe ≤ 400 µm", "LFP→LFP same probe, farther", "LFP→LFP same mouse, other probe", "LFP→LFP other mouse, same area",
     "LFP→LFP other mouse, same class", "LFP→LFP other mouse, other class", "LFP→EEG same mouse", "LFP→EEG other mouse",
     "EEG→EEG same mouse", "EEG→EEG other mouse", "EEG→LFP same mouse", "EEG→LFP other mouse"], "?")
T2 = P.pivot_table(index="pair", columns=["state", "grp"], values="score", aggfunc="median")
order = ["LFP→LFP same probe ≤ 400 µm", "LFP→LFP same probe, farther", "LFP→LFP same mouse, other probe", "LFP→LFP other mouse, same area",
         "LFP→LFP other mouse, same class", "LFP→LFP other mouse, other class", "LFP→EEG same mouse", "LFP→EEG other mouse",
         "EEG→EEG same mouse", "EEG→EEG other mouse", "EEG→LFP same mouse", "EEG→LFP other mouse"]
T2 = T2.reindex(order); T2.to_csv(OUT / "U2_transfer.csv")
own = U2["own"].groupby(["state", "kind"])[["own library", "tilt", "random smooth"]].median()
fig, ax = plt.subplots(figsize=(7.5, 4.6))
y = np.arange(len(order))[::-1]
for st, mk in (("awake", "o"), ("iso", "s")):
    for g, col in (("discovery", DISC_C), ("confirmation", CONF_C)):
        ax.plot(T2[(st, g)].to_numpy(), y, mk, color=col, mfc=col if st == "awake" else "white", ms=7,
                label=f"{g}, {'awake' if st == 'awake' else 'isoflurane'}")
ax.set_yticks(y, order, fontsize=8); ax.set_xlabel("captured share of the target's main fluctuation (median)")
ax.axvline(own.loc[("awake", "LFP"), "own library"], color="0.4", ls="--", lw=0.8)
ax.axvline(own.loc[("awake", "LFP"), "random smooth"], color="0.7", ls=":", lw=0.8)
ax.text(own.loc[("awake", "LFP"), "own library"], -0.9, "own motifs", fontsize=7, color="0.3", ha="right")
ax.text(own.loc[("awake", "LFP"), "random smooth"], -0.9, "random", fontsize=7, color="0.5")
ax.legend(frameon=False, fontsize=7, loc="lower left")
ax.set_title("U2: motifs transfer across mice and regions; the modality gap is the largest", fontsize=9, loc="left")
save(fig, "U2_transfer"); plt.show()
display(own.round(3)); T2.round(3)

own library   tilt  random smooth
state kind                                   
awake EEG         0.949  0.416          0.490
      LFP         0.962  0.490          0.630
iso   EEG         0.576  0.085          0.409
      LFP         0.899  0.340          0.551

state                                    awake                         \
grp                              across groups confirmation discovery   
pair                                                                    
LFP→LFP same probe ≤ 400 µm                NaN        0.903     0.906   
LFP→LFP same probe, farther                NaN        0.884     0.902   
LFP→LFP same mouse, other probe            NaN        0.888     0.893   
LFP→LFP other mouse, same area           0.885        0.872     0.887   
LFP→LFP other mouse, same class          0.881        0.879     0.878   
LFP→LFP other mouse, other class         0.879        0.878     0.876   
LFP→EEG same mouse                         NaN        0.725     0.828   
LFP→EEG other mouse                      0.772        0.722     0.824   
EEG→EEG same mouse                         NaN        0.910     0.910   
EEG→EEG other mouse                      0.780        0.817     0.785   
EEG→LFP same mouse                         NaN        0.754     0.860   
EEG→LFP other mouse                      0.789        0.769     0.820   

state                                      iso                         
grp                              across groups confirmation discovery  
pair                                                                   
LFP→LFP same probe ≤ 400 µm                NaN        0.889     0.912  
LFP→LFP same probe, farther                NaN        0.882     0.910  
LFP→LFP same mouse, other probe            NaN        0.886     0.898  
LFP→LFP other mouse, same area           0.892        0.870     0.894  
LFP→LFP other mouse, same class          0.888        0.879     0.893  
LFP→LFP other mouse, other class         0.885        0.879     0.887  
LFP→EEG same mouse                         NaN        0.877     0.908  
LFP→EEG other mouse                      0.889        0.877     0.909  
EEG→EEG same mouse                         NaN        0.588     0.804  
EEG→EEG other mouse                      0.690        0.599     0.807  
EEG→LFP same mouse                         NaN        0.667     0.791  
EEG→LFP other mouse                      0.719        0.656     0.822

## U3. Physiological grounding
Per unit, the nearest local (CSD) channel on its own probe; each motif's expression there (motifs learned without
the unit's mouse). Measures per probe: excess |r| with the population-removed rate over circular shifts; the same
after removing spectral tilt, the probe's band PC1 and five named bands (unique coupling); signed r with the raw rate
and with the population-removed rate; narrow- minus broad-spiking signed r.

In [7]:
S3 = ag.load(PU / "u3_summary.pkl"); G3, S = S3["G"], S3["S"]
rows = []
for st in ("awake", "isoflurane"):
    for m in (1, 2, 3):
        for f in ("signed_raw", "signed_pop", "narrow_minus_broad"):
            dd = G3[(G3.grp == "discovery") & (G3.state == st) & (G3.motif == m)][f].dropna()
            v = G3[(G3.grp == "confirmation") & (G3.state == st) & (G3.motif == m)][f].dropna()
            p = wilcoxon(v, alternative="greater" if dd.mean() > 0 else "less").pvalue
            rows.append({"state": st, "motif": NAMES[m - 1], "measure": f, "discovery mean": dd.mean(),
                         "discovery probes same sign": f"{(np.sign(dd) == np.sign(dd.mean())).sum()}/{len(dd)}",
                         "confirmation mean": v.mean(),
                         "confirmation probes same sign": f"{(np.sign(v) == np.sign(dd.mean())).sum()}/{len(v)}",
                         "p (one-sided)": p})
C3 = pd.DataFrame(rows); C3.to_csv(OUT / "U3_signed_confirmation.csv", index=False)
S.to_csv(OUT / "U3_coupling.csv", index=False)
S[["grp", "state", "motif", "probes", "excess|r|", "p", "excess|partial|", "p_partial"]].round(3)

,grp,state,motif,probes,excess|r|,p,excess|partial|,p_partial
0,confirmation,awake,1,12,0.064,0.01,-0.001,0.762
1,confirmation,awake,2,12,0.054,0.01,0.002,0.040
2,confirmation,awake,3,12,0.038,0.01,0.002,0.040
3,confirmation,isoflurane,1,12,0.049,0.01,0.004,0.010
4,confirmation,isoflurane,2,12,0.031,0.01,0.005,0.010
5,confirmation,isoflurane,3,12,0.034,0.01,0.005,0.010
6,discovery,awake,1,17,0.062,0.01,-0.005,1.000
7,discovery,awake,2,17,0.042,0.01,-0.003,1.000
8,discovery,awake,3,17,0.045,0.01,-0.003,0.960
9,discovery,isoflurane,1,14,0.061,0.01,0.001,0.149


In [8]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, f, lab in ((axs[0], "signed_raw", "r with raw rate"), (axs[1], "signed_pop", "r with population-removed rate")):
    for i, m in enumerate((1, 2, 3)):
        for j, st in enumerate(("awake", "isoflurane")):
            for g, col, off in (("discovery", DISC_C, -0.1), ("confirmation", CONF_C, 0.1)):
                v = G3[(G3.motif == m) & (G3.state == st) & (G3.grp == g)][f]
                x = i * 2.5 + j + off
                ax.plot(np.full(len(v), x) + np.random.default_rng(0).uniform(-0.05, 0.05, len(v)), v, "o", ms=3.5,
                        color=col, mfc=col if st == "awake" else "white", alpha=0.8)
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_xticks([i * 2.5 + j for i in range(3) for j in range(2)],
                  [f"{n.split()[0]}\n{'awake' if j == 0 else 'iso'}" for n in NAMES for j in range(2)], fontsize=8)
    ax.set_title(lab, fontsize=9, loc="left")
axs[0].set_ylabel("mean signed r over a probe's units")
axs[0].plot([], [], "o", color=DISC_C, label="discovery"); axs[0].plot([], [], "o", color=CONF_C, label="confirmation")
axs[0].legend(frameon=False, fontsize=7)
fig.suptitle("U3: each motif's sign of coupling with local firing, one dot per probe", fontsize=9, x=0.02, ha="left")
save(fig, "U3_signs"); plt.show()
C3.round(4)

,state,motif,measure,discovery mean,discovery probes same sign,confirmation mean,confirmation probes same sign,p (one-sided)
0,awake,M1 broadband,signed_raw,0.1156,16/17,0.0595,11/12,0.0005
1,awake,M1 broadband,signed_pop,0.0531,17/17,0.0470,12/12,0.0002
2,awake,M1 broadband,narrow_minus_broad,0.0492,11/17,-0.0044,5/12,0.6890
3,awake,M2 low vs gamma,signed_raw,0.0408,14/17,0.0499,11/12,0.0005
4,awake,M2 low vs gamma,signed_pop,0.0176,15/17,0.0191,11/12,0.0005
5,awake,M2 low vs gamma,narrow_minus_broad,-0.0233,10/17,0.0076,4/12,0.7407
6,awake,M3 slow vs beta,signed_raw,-0.0877,17/17,-0.0426,12/12,0.0002
7,awake,M3 slow vs beta,signed_pop,-0.0388,17/17,-0.0347,12/12,0.0002
8,awake,M3 slow vs beta,narrow_minus_broad,-0.0456,14/17,-0.0047,6/12,0.4849
9,isoflurane,M1 broadband,signed_raw,0.2563,14/14,0.2424,12/12,0.0002


## U4. Do the awake motifs describe isoflurane?
The awake set's held-out reconstruction under isoflurane, and the dominant direction of every channel under
isoflurane, clustered like U1 and matched to the awake motifs (|cos|).

In [9]:
U4 = ag.load(PU / "u4.pkl"); R4 = U4["reconstruction"]
R4["group"] = np.where(R4.mouse.isin(SP["discovery"]), "discovery", "confirmation"); R4["kind"] = np.where(R4.is_eeg, "EEG", "LFP")
rec4 = R4.groupby(["group", "state", "kind"])[["dictionary", "pooled PCA", "random smooth"]].median()
rec4.to_csv(OUT / "U4_reconstruction.csv")
pd4 = pd.DataFrame({f"{st}, {g}": cos for (st, g), (_, cos) in U4["principal_dictionaries"].items()}, index=NAMES).T
display(rec4.round(3)); pd4.round(3)

dictionary  pooled PCA  random smooth
group        state kind                                       
confirmation awake EEG        0.921       0.893          0.750
                   LFP        0.954       0.958          0.846
             iso   EEG        0.983       0.989          0.834
                   LFP        0.971       0.978          0.858
discovery    awake EEG        0.938       0.934          0.800
                   LFP        0.941       0.948          0.830
             iso   EEG        0.982       0.987          0.788
                   LFP        0.961       0.971          0.822

,M1 broadband,M2 low vs gamma,M3 slow vs beta
"awake, discovery",0.996,0.938,0.666
"awake, confirmation",0.998,0.780,0.697
"iso, discovery",0.991,0.574,0.558
"iso, confirmation",0.992,0.478,0.538


## U5. EEG motif → neurons beneath
|r| of each EEG electrode's motif expression with each unit's population-removed rate: units within 1 mm horizontally
(CCF anterior–posterior / medial–lateral) vs ≥ 3 mm. Shift null; p two-sided for beneath − far.

In [10]:
U5 = ag.load(PU / "u5_summary.pkl"); U5["motif"] = U5.motif.map(lambda m: NAMES[m - 1])
U5.to_csv(OUT / "U5_eeg_beneath.csv", index=False); U5.round(3)

,grp,state,motif,sessions,beneath,far,null,beneath-far,p_two
0,confirmation,awake,M1 broadband,3,0.118,0.131,0.054,-0.013,0.010
1,confirmation,awake,M2 low vs gamma,3,0.154,0.166,0.053,-0.011,0.010
2,confirmation,awake,M3 slow vs beta,3,0.141,0.131,0.049,0.009,0.010
3,confirmation,isoflurane,M1 broadband,3,0.116,0.113,0.042,0.003,0.010
4,confirmation,isoflurane,M2 low vs gamma,3,0.114,0.116,0.043,-0.002,0.752
5,confirmation,isoflurane,M3 slow vs beta,3,0.105,0.103,0.040,0.002,0.059
6,discovery,awake,M1 broadband,5,0.157,0.162,0.068,-0.004,0.366
7,discovery,awake,M2 low vs gamma,5,0.128,0.142,0.069,-0.014,0.010
8,discovery,awake,M3 slow vs beta,5,0.133,0.134,0.067,-0.001,0.762
9,discovery,isoflurane,M1 broadband,4,0.151,0.145,0.055,0.006,0.010


## Conclusions
- **One motif is universal in shape: M1, broadband** (all bands rise and fall together). It is the most common
  motif in every mouse, region class and modality, and an independent fit on the confirmation mice reproduces it
  (|cos| 0.999, p = 0.0005). It is the axis that dominates under isoflurane too.
- **M2, low (1–20 Hz) against gamma (35–80 Hz)**, close to spectral tilt (cos 0.71): the confirmation fit finds a
  similar axis (|cos| 0.89, crossover shifted up to ~40 Hz), but this does not beat random smooth shapes
  (p = 0.14 per motif; 0.45 with the stricter pre-set test). **M3, slow (1–9 Hz) against beta (10–30 Hz)**, does not
  reproduce in shape (|cos| 0.42). With 10 smooth bands, shapes below ~0.9 are hard to tell from chance.
- **The motifs are the principal axes of band covariance.** On held-out mice they capture 0.94–0.97 (LFP) and
  0.90–0.98 (EEG) of each channel's best possible, the same as pooled band PCA and well above random smooth
  directions (~0.8). The tracker's motifs are a reliable way to find these axes, not extra information beyond them.
- **Transfer (U2):** a motif from any LFP channel captures ~0.88–0.90 of another LFP channel's main fluctuation,
  almost regardless of mouse, region or distance (same probe nearby 0.90, other mouse other region 0.88; own motifs
  0.96; random 0.63). The largest loss is modality: LFP motifs explain the awake EEG less well (0.72–0.83), and the
  EEG's own awake motifs fail under isoflurane (0.58 vs 0.95 awake), where LFP motifs fit the EEG better (0.88–0.91).
- **Grounding (U3), confirmed in held-out mice, both states, nearly every probe:**
  - M1 broadband up → local firing up (raw and population-removed; under isoflurane much stronger on raw rates).
  - M2 toward gamma → firing up (population-removed: awake p = 0.0005, isoflurane p = 0.008).
  - M3 toward slow (delta/theta) and away from beta → firing down (every probe in both groups and states, p = 0.0002).
  - The discovery shapes of M2 and M3 predict spiking in new mice with the same sign even though their exact shapes
    do not reproduce: the relationship is robust to the precise spectral boundary.
  - None of the motifs adds coupling beyond spectral tilt, band PC1 and named bands (unique excess ≈ 0). Grounding is
    shared with standard spectral descriptors, not specific to the motif shape.
  - No cell-type signature: narrow- vs broad-spiking coupling differs inconsistently (one isoflurane p = 0.04).
- **EEG → beneath (U5):** EEG motif coupling with neurons is as strong 3 mm away as directly beneath. EEG motifs
  carry brain-state information, not local neuron information, at this resolution.

**Proposed universal set**, with physiological reading: (1) broadband power ↔ local firing (persists across mice,
regions, LFP/EEG and states); (2) gamma-vs-low tilt ↔ firing (awake strongest; the 1/f-slope / excitation-inhibition
axis); (3) slow-vs-beta ↔ firing down (robust in sign, not in shape). Caveats: 6–19 min of awake data per session; the
p-value floor is 0.01 for the shift tests (100 shifts); EEG sessions per group are 3–5.